# Lab Assignment 13

**Prerequisite:** the class demo notebook *ICL_and_Instruction_Finetuning_Demo.ipynb*  
**Runtime:** Google Colab with a T4 GPU (about 10–15 minutes of GPU time per task)

Each task is a small, real NLP problem: you design the solution, run the experiments and explain what you found. Only data loading and evaluation functions are given, so that everyone's numbers are comparable.

## Rules
- Use only the models named in each task.
- You may reuse any code from the class demo. Cite any other source.
- Fix random seeds and report the settings you used.
- The two tasks are independent, so you can restart the runtime between them.

## Submission
Submit this notebook as `.ipynb` **with all outputs visible**.

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate sentence-transformers scikit-learn

In [ ]:
import re, time, random, gc, sqlite3, warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq,
                          Trainer, TrainingArguments)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_causal_lm(name):
    """Load a decoder-only model in fp32 (numerically safe on a T4) and its tokenizer."""
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
    return tok, model


@torch.no_grad()
def generate_batch(model, tok, prompts, max_new_tokens=64, batch_size=8, stop_strings=None, desc="generating"):
    """Greedy, batched generation. Returns only the new text for each prompt, cut at the first stop string."""
    kwargs = dict(stop_strings=stop_strings, tokenizer=tok) if stop_strings else {}
    old_side, tok.padding_side = tok.padding_side, "left"
    outputs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc, leave=False):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, **kwargs)
        for row in gen[:, enc["input_ids"].shape[1]:]:
            text = tok.decode(row, skip_special_tokens=True)
            for s in (stop_strings or []):
                text = text.split(s)[0]
            outputs.append(text.strip())
    tok.padding_side = old_side
    return outputs

---
# Section A: In-Context Learning

## Task 1. A news-topic classifier with zero training (50 marks)

**Scenario.** A news website wants to tag articles as **World**, **Sports**, **Business** or **Sci/Tech**. It has only 200 labelled articles and no budget to train a model. Build the best classifier you can using **only in-context learning**, then decide whether it is actually a better choice than a simple classical model.

**Constraints**
- Model: `Qwen/Qwen2.5-0.5B` (base). No weight updates.
- Demonstrations may only come from `ag_pool`, with **at most 8 per prompt**.
- Every prediction must be one of the four classes. Report final numbers on `ag_test` using `report_classification`.

**What you must do**
1. **Baselines.** Evaluate zero-shot and 8 randomly chosen demonstrations.
2. **Retrieval-based demonstrations.** For each test article, choose the 8 most similar articles from `ag_pool` as demonstrations, using sentence embeddings from `sentence-transformers/all-MiniLM-L6-v2`. Compare with random selection.
3. **Classical baseline.** Train TF-IDF + logistic regression on the same 200 articles.
4. **Error analysis.** Show the confusion matrix of your best system and explain 3 of its mistakes.

**Analysis questions** (about 250 words in total)
- (a) Did retrieval-based demonstrations beat random ones? Why do you think similar examples help?
- (b) Which classes get confused most often, and why might that be?
- (c) Compare your best ICL system with logistic regression on accuracy **and** speed (seconds per article). Which would you deploy for this website, and why?



In [ ]:
ag = load_dataset("fancyzhx/ag_news")
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]

ag_test = ag["test"].shuffle(seed=0).select(range(200))
_train = ag["train"].shuffle(seed=1)
ag_pool = _train.select(range(200))          # the ONLY labelled data you may use
ag_dev = _train.select(range(200, 300))      # use for design decisions (prompt format, label words)

for name, ds in [("pool", ag_pool), ("dev", ag_dev), ("test", ag_test)]:
    counts = Counter(ds["label"])
    print(f"{name:5s}", {AG_LABELS[k]: counts[k] for k in range(4)})
print("\nExample:", ag_pool[0])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

RESULTS_T1 = []   # append the dict returned by report_classification for every system


def report_classification(y_true, y_pred, method, sec_per_example=None, show_cm=False):
    """Fixed evaluation for Task 1. y_true / y_pred are lists of label ids (0-3)."""
    res = {"method": method,
           "accuracy": accuracy_score(y_true, y_pred),
           "macro_f1": f1_score(y_true, y_pred, average="macro"),
           "sec_per_example": sec_per_example}
    if show_cm:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=list(range(4)), display_labels=AG_LABELS,
                                                cmap="Blues", xticks_rotation=45)
        plt.title(method); plt.show()
    return res

### Task 1: Your solution
Add as many code and text cells as you need.

In [ ]:
# Your code here

### Task 1: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._

---
# Section B: Instruction Fine-Tuning

## Task 2. English → SQL: prompt a ready-made model or fine-tune your own? (50 marks)

**Scenario.** A company wants its analysts to query databases in plain English. Given a table schema (a `CREATE TABLE` statement) and a question, a model must write the SQL query. Should the company **prompt** an existing instruction-tuned model, or **fine-tune** a small model on examples? Build both and recommend one.

**Data:** each example has a `question`, a `context` (the schema) and an `answer` (the gold SQL).

**Constraints**
- Prompting system: `Qwen/Qwen2.5-0.5B-Instruct` (use its chat template).
- Fine-tuned system: `openai-community/gpt2` (124M), with the loss computed on the SQL tokens only.
- Training and demonstration data come only from `sql_train_pool`. Evaluate every system on `sql_test` with `evaluate_sql`.

**What you must do**
1. **Prompting baseline.** Prompt Qwen2.5-0.5B-Instruct with 3 examples from the pool. The model may add explanations or code fences, so extract just the SQL from its output.
2. **Fine-tuning.** Design an input format for (schema, question) → SQL and fine-tune GPT-2 twice, on **500** and on **2,000** examples, with the same hyperparameters.
3. **Error analysis.** Look at 10 mistakes of your best system and label each one: wrong column, wrong condition or aggregate, wrong value, invalid SQL, or correct but written differently.

**Analysis questions** (about 250 words in total)
- (a) How did fine-tuned GPT-2 compare with prompting a model four times its size? How much did going from 500 to 2,000 examples help?
- (b) How many of your "mistakes" were actually correct queries written differently? Why does exact match underestimate performance?
- (c) Write a short recommendation to the company: prompting or fine-tuning? Consider accuracy, the labelled data needed, cost, and what happens when new tables are added.

**Marking:** prompting baseline 12 · fine-tuning (both sizes) 18 · error analysis 8 · discussion 12

In [ ]:
sql_ds = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=0)
sql_test = sql_ds.select(range(200))
sql_train_pool = sql_ds.select(range(200, 5200))   # training, validation and demonstration data

ex = sql_test[0]
print("SCHEMA  :", ex["context"]); print("QUESTION:", ex["question"]); print("GOLD SQL:", ex["answer"])

In [ ]:
def normalize_sql(q):
    """Lenient normalisation for exact match: case, whitespace, quote style, trailing semicolon."""
    q = q.strip().rstrip(";").replace('"', "'")
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"\s*([(),=<>])\s*", r"\1", q)
    return q.lower().strip()


def executes(sql, context):
    """True if the query runs without error on an empty database built from the schema."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(context)
        con.execute(sql).fetchall()
        return True
    except Exception:
        return False
    finally:
        con.close()


def evaluate_sql(preds, dataset, system):
    """Fixed evaluation for Task 2. preds: list of SQL strings aligned with dataset."""
    golds, contexts = dataset["answer"], dataset["context"]
    return {"system": system,
            "exact_match": float(np.mean([normalize_sql(p) == normalize_sql(g) for p, g in zip(preds, golds)])),
            "executable": float(np.mean([executes(p, c) for p, c in zip(preds, contexts)]))}


RESULTS_T2 = []   # append the dict returned by evaluate_sql for every system
print("Reference (gold queries):", evaluate_sql(sql_test["answer"], sql_test, "gold"))

### Task 2: Your solution
Add as many code and text cells as you need.

In [ ]:
# Your code here

### Task 2: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._